In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [2]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip imdb-dataset-of-50k-movie-reviews.zip

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:00<00:00, 100MB/s] 

Archive:  imdb-dataset-of-50k-movie-reviews.zip
  inflating: IMDB Dataset.csv        


In [3]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
df = pd.read_csv('IMDB Dataset.csv')
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [5]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [6]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.discard('not')
stop_words.discard('no')
stop_words.discard('never')

In [7]:
import re
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()

    # keep negation words
    text = re.sub(r"n't", " not", text)

    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [8]:
print(df.columns)

Index(['review', 'sentiment'], dtype='object')


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,      # increase features
    ngram_range=(1,2),       # tigrams
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X = tfidf.fit_transform(df['review'])
y = df['sentiment']

In [10]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

lr = LogisticRegression()

params = {
    'C': [0.1, 0.5, 1, 2, 4, 6, 8],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear', 'saga'],
    'max_iter': [1000, 1500, 2000, 3000],
    'class_weight': [None, 'balanced']
}

random = RandomizedSearchCV(
    lr,
    param_distributions=params,
    n_iter=15,          # increased search
    cv=3,
    scoring='accuracy',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random.fit(x_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


RandomizedSearchCV(cv=3, estimator=LogisticRegression(), n_iter=15, n_jobs=-1,
                   param_distributions={'C': [0.1, 0.5, 1, 2, 4, 6, 8],
                                        'class_weight': [None, 'balanced'],
                                        'max_iter': [1000, 1500, 2000, 3000],
                                        'penalty': ['l2'],
                                        'solver': ['lbfgs', 'liblinear',
                                                   'saga']},
                   random_state=42, scoring='accuracy', verbose=2)

In [12]:
best_model = random.best_estimator_
print("Best Parameters:", random.best_params_)

Best Parameters: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 3000, 'class_weight': None, 'C': 4}


In [13]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = best_model.predict(x_test)
y_train_pred = best_model.predict(x_train)

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Train accuracy: 0.96195
Accuracy: 0.9115
              precision    recall  f1-score   support

           0       0.92      0.91      0.91      4961
           1       0.91      0.92      0.91      5039

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000



In [20]:
user_input = input("Enter your movie review: ")

clean = [clean_text(user_input)]
vec = tfidf.transform(clean)

pred = best_model.predict(vec)

if pred[0] == 1:
    print("Positive 😊")
else:
    print("Negative 😞")

Enter your movie review: When life gives you lemons, well, you become a football star.
Positive 😊
